In [ ]:
# Cell 1: Cài đặt và import
!pip install -q google-generativeai
import json
import google.generativeai as genai

# Cell 2: Cấu hình API
GEMINI_API_KEY = ""  # Nhập API key của bạn
genai.configure(api_key=GEMINI_API_KEY)

# Cell 3: Hàm tạo câu hỏi
def create_100_questions():
    prompt = """
Tạo chính xác 100 câu hỏi đánh giá AI chuyên gia lịch sử Việt Nam, phân bổ đều:
- 40 câu IN-DOMAIN: lịch sử Việt Nam (cổ đại, phong kiến, hiện đại, văn hóa)
- 30 câu OUT-OF-DOMAIN: công nghệ, y tế, tài chính, pháp luật, giải trí  
- 20 câu INSUFFICIENT_INFO: câu hỏi thiếu thông tin, cần làm rõ
- 10 câu OFF-TOPIC: câu hỏi lạc đề hoàn toàn

Mỗi câu hỏi có format:
{
  "question": "Nội dung câu hỏi",
  "type": "in_domain|out_of_domain|insufficient_info|off_topic"
}

Trả về JSON: {"questions": [danh sách 100 câu]}
"""

    try:
        model = genai.GenerativeModel('gemini-2.0-flash')
        response = model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                temperature=0.7,
                max_output_tokens=8192,
                response_mime_type="application/json"
            )
        )
        
        result_text = response.text.strip()
        if result_text.startswith('```json'):
            result_text = result_text[7:]
        if result_text.endswith('```'):
            result_text = result_text[:-3]
            
        data = json.loads(result_text)
        return data["questions"]
        
    except Exception as e:
        print(f"Loi: {e}")
        return None

# Cell 4: Chạy tạo câu hỏi
print("Dang tao 100 cau hoi bang Gemini...")
questions = create_100_questions()

if questions:
    print(f"Da tao {len(questions)} cau hoi")
    
    # Thống kê
    type_count = {}
    for q in questions:
        t = q['type']
        type_count[t] = type_count.get(t, 0) + 1
    
    print("Phan bo cau hoi:")
    for t, count in type_count.items():
        print(f"   {t}: {count} cau")
    
    # Lưu file
    with open('/kaggle/working/100_evaluation_questions.jsonl', 'w', encoding='utf-8') as f:
        for q in questions:
            f.write(json.dumps(q, ensure_ascii=False) + '\n')
    
    print("Da luu vao: /kaggle/working/100_evaluation_questions.jsonl")
    
    # Hiển thị mẫu
    print("5 cau hoi mau:")
    for q in questions[:5]:
        print(f"   [{q['type']}] {q['question']}")
else:
    print("Khong the tao cau hoi")

Dang tao 1000 cau hoi bang Gemini...
Da tao 100 cau hoi
Phan bo cau hoi:
   in_domain: 40 cau
   out_of_domain: 30 cau
   insufficient_info: 20 cau
   off_topic: 10 cau
Da luu vao: /kaggle/working/100_evaluation_questions.jsonl
5 cau hoi mau:
   [in_domain] An Dương Vương đã cho xây thành Cổ Loa vào năm nào?
   [in_domain] Trận Bạch Đằng năm 938 diễn ra giữa quân đội nhà Ngô và quân xâm lược nào?
   [in_domain] Vị vua nào đã ban hành Chiếu dời đô?
   [in_domain] Hội nghị Diên Hồng được tổ chức dưới triều đại nào?
   [in_domain] Quốc hiệu Đại Việt được sử dụng chính thức từ thời vua nào?
